# Operations Research Group Project

## Scope

**Primal Problem (recap)**:

$$\min_{x} \quad Z = \sum_{i \in I} c_i x_i$$

**Dual Problem**:

$$\max_{y} \quad W = \mathbf{b}_{ub}^\top \mathbf{y} + \mathbf{x}_{ub}^\top \boldsymbol{\lambda}_{ub}$$

**Dual Variables**:

$y_k$ — shadow price of constraint $k$ in $A_{ub}\,\mathbf{x} \leq \mathbf{b}_{ub}$

$\lambda_{ub,i}$ — shadow price of the portion cap $x_i \leq x_{ub}$

**Shadow Price Interpretation**:

For a binding lower-bound constraint (nutrient minimum $\geq b_j^{LB}$, encoded as $-A_j x \leq -b_j^{LB}$):

$$\text{shadow\_price}_j = -y_k > 0$$

A shadow price of $s$ means: if the nutrient minimum $b_j^{LB}$ increases by 1 unit, the minimum lunch cost $Z^*$ increases by $s$ USD.

**Strong Duality Property**:

$$Z^* = W^* \quad \text{(at optimality)}$$

**Complementary Slackness**:

$$y_k > 0 \implies \text{constraint } k \text{ is binding (slack} = 0)$$

$$\text{slack}_k > 0 \implies y_k = 0$$

**Constraint Index Sets**:

$i \in I = \{1, 2, \ldots, 204\}$ — food items

$j \in J = \{1, 2, \ldots, 29\}$ — nutrients

$k \in K = \{1, 2, \ldots, 28\}$ — LP constraint rows (assembled from $J_{\min}$, $J_{\max}$, $J_{\text{range}}$)

In [ ]:
# Necessary imports
import numpy as np
import pandas as pd

from scipy.optimize import linprog, OptimizeResult
from tabulate import tabulate

import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Data loc
PATH_PRICES    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/food_prices_200.xlsx"
PATH_NUTRITION = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/food_nutrition_200.xlsx"
PATH_REQ_LB    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/dri_lunch_lb.csv"
PATH_REQ_UB    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/dri_lunch_ub.csv"

# Portion cap
MAX_GRAMS_PER_FOOD = 200.0

In [ ]:
# Data load
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Loading all required data

    Tap water should be excluded

    Returns:
    nutrition   : (204, 35) - food nutritional content per 100g
    prices      : (204, 13) - food prices per gram
    lb          : (5, 29+4) - lower bounds per age-sex group
    ub          : (5, 29+4) - upper bounds per age-sex group
    """
    nutrition = pd.read_excel(PATH_NUTRITION, keep_default_na=False)
    prices    = pd.read_excel(PATH_PRICES,    keep_default_na=False)
    lb        = pd.read_csv(PATH_REQ_LB).set_index("age_sex_group")
    ub        = pd.read_csv(PATH_REQ_UB).set_index("age_sex_group")

    # Remove water
    nutrition = nutrition[nutrition["food_name"] != "Water, Tap"].reset_index(drop=True)
    prices    = prices[prices["food_name"] != "Water, Tap"].reset_index(drop=True)

    return nutrition, prices, lb, ub

In [ ]:
nutrition, prices, lb, ub = load_data()
print(f"Foods: {len(nutrition)}")
print(f"Groups: {list(lb.index)}")

In [ ]:
# Build LP coefficient matrices
def build_matrices(
        nutrition: pd.DataFrame,
        prices:    pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, list[str], list[str]]:
    """
    Constructing the obj vec c and const coeff matrix A

    Notation:
    n   : number of foods
    m   : number of nutrients
    c   : cost per gram (USD/g)
    A   : nutrient per gram

    Returns:
    c           : (n,) - obj coeff (USD per gram)
    A           : (n, m) - nutrient per gram matrix
    food_names  : list[str]
    nut_cols    : list[str]
    """
    nut_cols = list(nutrition.columns[6:])  # 29 nutrients

    merged = nutrition.merge(
        prices[["food_id", "price_per_1g_usd"]],
        on="food_id",
        how="inner"
    )

    food_names = list(merged["food_name"])
    c          = merged["price_per_1g_usd"].to_numpy(dtype=float)
    A          = merged[nut_cols].to_numpy(dtype=float) / 100.0

    return c, A, food_names, nut_cols

In [ ]:
c, A, food_names, nut_cols = build_matrices(nutrition, prices)
print(f"Obj vector c : {c.shape}     (USD/g)")
print(f"Const mat A  : {A.shape}     (nutrient/g)")
print(f"Cost range   : ${c.min():.4f} - ${c.max():.4f} per gram")

In [ ]:
# Build constraints
def get_constraints() -> dict[str, list[str]]:
    """
    Classifying each nutrient into LP constraint types

    Categories:
    min_only    : sum_i a_ij x_i >= LB_j   (RDA/AI minimum requirements)
    max_only    : sum_i a_ij x_i <= UB_j   (NSLP/DGA ceilings)
    range       : LB_j <= sum_i a_ij x_i <= UB_j  (energy + fat)
    skip        : no numeric DRI -- excluded from LP constraints
    """
    min_only = [
        "protein_g",
        "carbohydrates_g",
        "dietary_fiber_g",
        "calcium_mg",
        "iron_mg",
        "magnesium_mg",
        "phosphorus_mg",
        "potassium_mg",
        "zinc_mg",
        "vitamin_a_ug_rae",
        "vitamin_d_ug",
        "vitamin_e_mg_ate",
        "vitamin_k_ug",
        "vitamin_c_mg",
        "thiamin_b1_mg",
        "riboflavin_b2_mg",
        "niacin_b3_mg",
        "vitamin_b6_mg",
        "folate_ug_dfe",
        "vitamin_b12_ug",
        "selenium_ug",
    ]
    max_only = [
        "sugars_g",
        "saturated_fat_g",
        "sodium_mg",
    ]
    range_nuts = [
        "energy_kcal",
        "total_fat_g",
    ]
    skip = [
        "monounsaturated_fat_g",
        "polyunsaturated_fat_g",
        "cholesterol_mg",
    ]

    return {
        "min_only" : min_only,
        "max_only" : max_only,
        "range"    : range_nuts,
        "skip"     : skip,
    }

In [ ]:
constraint_sets = get_constraints()
for ctype, nuts in constraint_sets.items():
    print(f"{ctype}: {len(nuts):2d} nutrients")

In [ ]:
# Build A_ub system and solve
def build_system(
        A:               np.ndarray,
        nut_cols:        list[str],
        lb_row:          pd.Series,
        ub_row:          pd.Series,
        constraint_sets: dict[str, list[str]],
) -> tuple[np.ndarray, np.ndarray, list[tuple]]:
    """
    Assembles the A_ub matrix, b_ub vector, and constraint name list
    needed for linprog and dual variable extraction.

    Encoding:
    min constraint  (Ax >= b)  -->  -Ax <= -b
    max constraint  (Ax <= b)  -->   Ax <=  b
    range lb        (Ax >= b)  -->  -Ax <= -b
    range ub        (Ax <= b)  -->   Ax <=  b

    Returns:
    A_ub    : (K, n) constraint matrix
    b_ub    : (K,)   constraint RHS
    names   : list of (constraint_type, nutrient, rhs_value)
    """
    nut_idx = {nut: j for j, nut in enumerate(nut_cols)}

    A_ub_rows, b_ub_rows, names = [], [], []

    def _valid(val) -> bool:
        if val == "": return False
        try: return not np.isnan(float(val))
        except: return False

    for nut in constraint_sets["min_only"]:
        j   = nut_idx[nut]
        val = lb_row.get(nut, "")
        if not _valid(val): continue
        A_ub_rows.append(-A[:, j]); b_ub_rows.append(-float(val))
        names.append(("min_lb", nut, float(val)))

    for nut in constraint_sets["max_only"]:
        j   = nut_idx[nut]
        val = ub_row.get(nut, "")
        if not _valid(val): continue
        A_ub_rows.append(A[:, j]); b_ub_rows.append(float(val))
        names.append(("max_ub", nut, float(val)))

    for nut in constraint_sets["range"]:
        j     = nut_idx[nut]
        v_lb  = lb_row.get(nut, "")
        v_ub  = ub_row.get(nut, "")
        if _valid(v_lb):
            A_ub_rows.append(-A[:, j]); b_ub_rows.append(-float(v_lb))
            names.append(("range_lb", nut, float(v_lb)))
        if _valid(v_ub):
            A_ub_rows.append(A[:, j]); b_ub_rows.append(float(v_ub))
            names.append(("range_ub", nut, float(v_ub)))

    return np.array(A_ub_rows), np.array(b_ub_rows), names

In [ ]:
# Solve LP and capture primal + dual
def solve_dual(
        group_id:        str,
        c:               np.ndarray,
        A:               np.ndarray,
        food_names:      list[str],
        nut_cols:        list[str],
        lb_row:          pd.Series,
        ub_row:          pd.Series,
        constraint_sets: dict[str, list[str]],
) -> dict:
    """
    Solve LP and extract both primal and dual information

    Dual variables (shadow prices) sourced from res.ineqlin.marginals
    and res.upper.marginals (scipy HiGHS)

    Strong duality:
    Z* = b_ub @ mu + x_ub * sum(lam_ub)

    Shadow price convention:
    shadow_price = -mu[k]  (positive for binding constraints)
    Interpretation: 1-unit tightening of constraint k raises Z* by shadow_price USD

    Returns:
    dict with keys: group_id, status, cost_usd, primal_x, dual_mu,
                    dual_lam_ub, A_ub, b_ub, con_names, n_foods
    """
    n = len(food_names)

    A_ub, b_ub, con_names = build_system(
        A, nut_cols, lb_row, ub_row, constraint_sets
    )
    bounds = [(0.0, MAX_GRAMS_PER_FOOD)] * n

    res: OptimizeResult = linprog(
        c      = c,
        A_ub   = A_ub,
        b_ub   = b_ub,
        bounds = bounds,
        method = "highs",
    )

    if res.status != 0:
        return {"group_id": group_id, "status": res.message}

    return {
        "group_id"  : group_id,
        "status"    : "Optimal",
        "cost_usd"  : round(float(res.fun), 6),
        "primal_x"  : res.x,
        "dual_mu"   : res.ineqlin.marginals,   # shadow prices for A_ub constraints
        "dual_lam_ub": res.upper.marginals,    # shadow prices for x <= x_ub bounds
        "A_ub"      : A_ub,
        "b_ub"      : b_ub,
        "con_names" : con_names,
        "n_foods"   : int((res.x > 0.1).sum()),
    }

In [ ]:
# Solve for all groups
dual_results = []

for group_id in lb.index:
    result = solve_dual(
        group_id        = group_id,
        c               = c,
        A               = A,
        food_names      = food_names,
        nut_cols        = nut_cols,
        lb_row          = lb.loc[group_id],
        ub_row          = ub.loc[group_id],
        constraint_sets = constraint_sets,
    )
    dual_results.append(result)
    print(f"{group_id:25s}  Z* = ${result['cost_usd']:.6f}")

In [ ]:
# Extract shadow prices mapped to nutrient names
def extract_shadow_prices(result: dict) -> pd.DataFrame:
    """
    Map raw dual variables to nutrient constraint names

    shadow_price = -mu[k]
    Positive shadow price: tightening constraint k by 1 unit raises Z* by shadow_price USD
    Zero shadow price: constraint k is non-binding (complementary slackness)

    Returns:
    DataFrame with columns:
        con_type, nutrient, rhs, mu, shadow_price, slack, binding
    """
    mu         = result["dual_mu"]
    A_ub       = result["A_ub"]
    b_ub       = result["b_ub"]
    x          = result["primal_x"]
    con_names  = result["con_names"]

    slack = b_ub - A_ub @ x   # slack[k] = b_ub[k] - A_ub[k] @ x

    rows = []
    for k, (ctype, nut, rhs) in enumerate(con_names):
        sp      = round(-mu[k], 8)
        sl      = round(slack[k], 6)
        binding = abs(sl) < 1e-4
        rows.append({
            "con_type"    : ctype,
            "nutrient"    : nut,
            "rhs"         : rhs,
            "mu"          : round(mu[k], 8),
            "shadow_price": sp,
            "slack"       : sl,
            "binding"     : binding,
        })

    return pd.DataFrame(rows)

In [ ]:
# Extract for all groups and attach to results
for result in dual_results:
    if result["status"] != "Optimal":
        continue
    result["shadow_df"] = extract_shadow_prices(result)

In [ ]:
# Verify strong duality: Z* = W*
def verify_strong_duality(result: dict) -> dict:
    """
    Verify strong duality property: Z* = W*

    W* = b_ub @ mu + x_ub @ lam_ub

    From the lecture:
    'If the Primal has an optimal solution, the Dual also has an optimal
    solution and their objective values are exactly equal: Z* = W*'

    Returns:
    dict with z_star, w_star, gap, verified
    """
    mu      = result["dual_mu"]
    lam_ub  = result["dual_lam_ub"]
    b_ub    = result["b_ub"]
    n       = len(result["primal_x"])

    z_star  = result["cost_usd"]
    w_star  = b_ub @ mu + MAX_GRAMS_PER_FOOD * np.ones(n) @ lam_ub

    return {
        "group_id" : result["group_id"],
        "z_star"   : round(z_star, 6),
        "w_star"   : round(float(w_star), 6),
        "gap"      : round(abs(z_star - float(w_star)), 8),
        "verified" : np.isclose(z_star, float(w_star), rtol=1e-4),
    }

In [ ]:
# Print strong duality verification for all groups
duality_rows = []
for result in dual_results:
    if result["status"] != "Optimal":
        continue
    v = verify_strong_duality(result)
    duality_rows.append([
        v["group_id"],
        f"${v['z_star']:.6f}",
        f"${v['w_star']:.6f}",
        f"{v['gap']:.2e}",
        "YES" if v["verified"] else "NO",
    ])

print(tabulate(
    duality_rows,
    headers=["Group", "Z* (primal)", "W* (dual)", "Gap", "Z* = W*"],
    tablefmt="rounded_outline",
))

In [ ]:
# Verify complementary slackness
def verify_complementary_slackness(result: dict) -> pd.DataFrame:
    """
    Verify complementary slackness theorem:

    From the lecture:
    'At the optimal solution, if a resource is not fully utilised (slack > 0),
    its shadow price must be exactly zero.'

    Violations: binding but zero shadow price, or non-binding but nonzero shadow price

    Returns:
    DataFrame showing binding status, shadow price, and CS check for each constraint
    """
    df   = result["shadow_df"].copy()
    ZERO = 1e-6

    # Complementary slackness conditions
    # 1) binding (slack ~= 0) AND shadow_price ~= 0  -> violation
    # 2) non-binding (slack > 0) AND shadow_price != 0 -> violation
    df["sp_nonzero"] = df["shadow_price"].abs() > ZERO
    df["cs_holds"]   = df["binding"] == df["sp_nonzero"]

    return df[["nutrient", "con_type", "slack", "shadow_price", "binding", "sp_nonzero", "cs_holds"]]

In [ ]:
# Print complementary slackness for all groups
for result in dual_results:
    if result["status"] != "Optimal":
        continue

    cs_df = verify_complementary_slackness(result)
    violations = (~cs_df["cs_holds"]).sum()
    satisfied  = cs_df["cs_holds"].sum()

    print(f"\nGROUP : {result['group_id'].upper().replace('_', ' ')}")
    print(f"    CS satisfied : {satisfied} / {len(cs_df)} constraints")
    print(f"    CS violations: {violations}")

    display_rows = []
    for _, row in cs_df.iterrows():
        display_rows.append([
            row["nutrient"],
            row["con_type"],
            f"{row['slack']:.6f}",
            f"{row['shadow_price']:.6f}",
            "YES" if row["binding"] else "no",
            "YES" if row["sp_nonzero"] else "no",
            "OK" if row["cs_holds"] else "VIOLATION",
        ])

    print(tabulate(
        display_rows,
        headers=["Nutrient", "Type", "Slack", "Shadow Price", "Binding", "SP != 0", "CS"],
        tablefmt="rounded_outline",
    ))

In [ ]:
# Print dual results summary per group
def print_dual_results(result: dict) -> None:
    """
    Print shadow prices and duality summary for one group

    Sections:
    1. Strong duality verification (Z* = W*)
    2. Active shadow prices (binding constraints and their cost impact)
    3. Most expensive nutrients (ranked by shadow price)
    """
    if result["status"] != "Optimal":
        return

    gid = result["group_id"]
    print(f"\nGROUP : {gid.upper().replace('_', ' ')}")
    print(f"    Z* = ${result['cost_usd']:.6f}")

    # Strong duality
    v = verify_strong_duality(result)
    print(f"    W* = ${v['w_star']:.6f}    Z* = W* : {'YES' if v['verified'] else 'NO'}")

    # Shadow prices table
    df = result["shadow_df"]
    active = df[df["shadow_price"].abs() > 1e-6].sort_values(
        "shadow_price", ascending=False
    )

    print(f"\n    Shadow prices (binding constraints only)")
    print(f"    Interpretation: +1 unit tightening of constraint raises Z* by shadow_price USD")

    rows = []
    for _, row in active.iterrows():
        rows.append([
            row["nutrient"],
            row["con_type"],
            f"{row['rhs']:.3f}",
            f"${row['shadow_price']:.6f}",
        ])

    print(tabulate(
        rows,
        headers=["Nutrient", "Constraint type", "RHS (LB or UB)", "Shadow price (USD/unit)"],
        tablefmt="rounded_outline",
    ))

In [ ]:
for result in dual_results:
    print_dual_results(result)

In [ ]:
# Cross-group shadow price comparison
def print_shadow_comparison(dual_results: list[dict]) -> None:
    """
    Compare shadow prices across all groups for the consistently binding nutrients

    Shows which nutrient constraint is the most expensive to tighten per group
    """
    # Collect all active nutrients across groups
    all_rows = []
    for result in dual_results:
        if result["status"] != "Optimal":
            continue
        df = result["shadow_df"]
        active = df[df["shadow_price"].abs() > 1e-6][["nutrient", "shadow_price"]]
        active = active.copy()
        active["group"] = result["group_id"]
        all_rows.append(active)

    combined = pd.concat(all_rows, ignore_index=True)
    pivot    = (
        combined
        .pivot_table(index="nutrient", columns="group", values="shadow_price", fill_value=0.0)
        .reset_index()
    )

    print("Shadow price comparison across groups (USD per unit tightening)")
    print(tabulate(
        pivot,
        headers="keys",
        tablefmt="rounded_outline",
        floatfmt=".6f",
        showindex=False,
    ))

In [ ]:
print_shadow_comparison(dual_results)

In [ ]:
# Save dual results to markdown
def save_dual_md(result: dict, save_path: str = ".") -> None:
    """
    Save duality analysis for one group as a Markdown file

    Output: dual_{group_id}.md
    """
    if result["status"] != "Optimal":
        print(f"  Skipping {result['group_id']} — no optimal solution.")
        return

    gid      = result["group_id"]
    filename = f"dual_{gid}.md"
    v        = verify_strong_duality(result)
    df       = result["shadow_df"]
    active   = df[df["shadow_price"].abs() > 1e-6].sort_values("shadow_price", ascending=False)

    lines = []

    # Header
    lines.append(f"# Duality Analysis — {gid.replace('_', ' ').title()}\n")
    lines.append(f"**Solver status:** {result['status']}  ")
    lines.append(f"**Z* (primal):** ${result['cost_usd']:.6f} USD  ")
    lines.append(f"**W* (dual):** ${v['w_star']:.6f} USD  ")
    lines.append(f"**Strong duality Z* = W*:** {'YES' if v['verified'] else 'NO'}\n")

    # Shadow prices
    lines.append("## Shadow Prices (Binding Constraints)\n")
    lines.append(
        "Shadow price = cost increase in Z* from tightening that constraint by 1 unit.  \n"
        "Zero shadow price = non-binding constraint (complementary slackness).\n"
    )
    lines.append("| Nutrient | Constraint type | RHS | Shadow price (USD/unit) |")
    lines.append("|----------|----------------|----:|------------------------:|")
    for _, row in active.iterrows():
        lines.append(
            f"| {row['nutrient']} "
            f"| {row['con_type']} "
            f"| {row['rhs']:.3f} "
            f"| ${row['shadow_price']:.6f} |"
        )
    lines.append("")

    # Complementary slackness
    cs_df = verify_complementary_slackness(result)
    lines.append("## Complementary Slackness Verification\n")
    lines.append("| Nutrient | Type | Slack | Shadow price | Binding | SP != 0 | CS holds |")
    lines.append("|----------|------|------:|-------------:|:-------:|:-------:|:--------:|")
    for _, row in cs_df.iterrows():
        lines.append(
            f"| {row['nutrient']} "
            f"| {row['con_type']} "
            f"| {row['slack']:.6f} "
            f"| {row['shadow_price']:.6f} "
            f"| {'YES' if row['binding'] else 'no'} "
            f"| {'YES' if row['sp_nonzero'] else 'no'} "
            f"| {'OK' if row['cs_holds'] else 'VIOLATION'} |"
        )
    lines.append("")

    # Write and save
    os.makedirs(save_path, exist_ok=True)
    filepath = os.path.join(save_path, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Saved: {filename}")

In [ ]:
SAVE_PATH = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/result/duality"
for result in dual_results:
    save_dual_md(result, save_path=SAVE_PATH)

## References

### LP Duality Theory

Hillier, F. S., & Lieberman, G. J. (2015).
*Introduction to operations research* (10th ed.). McGraw-Hill Education.

Vanderbei, R. J. (2020).
*Linear programming: Foundations and extensions* (5th ed.). Springer.
https://doi.org/10.1007/978-3-030-39415-8

### LP Solver

Huangfu, Q., & Hall, J. A. J. (2018).
Parallelizing the dual revised simplex method.
*Mathematical Programming Computation*, 10(1), 119–142.
https://doi.org/10.1007/s12532-017-0130-5

Virtanen, P., Gommers, R., Oliphant, T. E., Haberland, M., Reddy, T.,
Cournapeau, D., … van der Walt, S. J. (2020).
SciPy 1.0: Fundamental algorithms for scientific computing in Python.
*Nature Methods*, 17, 261–272.
https://doi.org/10.1038/s41592-019-0686-2

### Nutritional Data

U.S. Department of Agriculture, Agricultural Research Service. (2019).
*FoodData Central* (SR Legacy Foods / Foundation Foods).
Retrieved from https://fdc.nal.usda.gov

### Food Price Data

U.S. Bureau of Labor Statistics. (2024).
*Average retail food prices — CPI average price series*.
Retrieved from https://data.bls.gov

### Dietary Reference Intakes

National Academies of Sciences, Engineering, and Medicine. (2019).
*Dietary reference intakes for sodium and potassium* (revised).
National Academies Press.
https://doi.org/10.17226/25353

U.S. Department of Agriculture & U.S. Department of Health and Human Services. (2020).
*Dietary guidelines for Americans 2020–2025* (9th ed.).
Retrieved from https://www.dietaryguidelines.gov

### School Nutrition Policy

U.S. Department of Agriculture, Food and Nutrition Service. (2024).
Child nutrition programs: Meal patterns consistent with the 2020–2025
dietary guidelines for Americans. *Federal Register*, 89(81).
Retrieved from https://www.federalregister.gov/documents/2024/04/25/2024-08098